# 03 - Probability vs Confidence

The two numbers Jev attaches to its answers are the foundation of every reliable system
built on it. They are related but not the same thing:

- **Probabilities** are the answer's distribution over *your* options or levels.
  They sum to 1. They tell you *what the model believes*.
- **Confidence** is a single 0-to-1 statistic computed **from** that distribution's shape.
  All probability piled on one outcome gives 1; an even spread gives 0.
  It tells you *whether to act*.

And the property that makes both usable: Jev is trained for **calibration**. Across many
predictions, outcomes it assigns 0.2 should happen about 20% of the time, and outcomes it
assigns 0.8 about 80%. (That is a statement about groups of predictions, never a guarantee
about one answer.) An uncalibrated model's 0.9 is just vibes; a calibrated model's 0.9 is an
input your code can price risk with.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Find the TypeSafe API key: environment first, then any .env file up the tree.
if "TYPESAFE_API_KEY" not in os.environ:
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / ".env").exists():
            load_dotenv(folder / ".env")
            break
assert os.environ.get("TYPESAFE_API_KEY"), "Set TYPESAFE_API_KEY (get a key at console.typesafe.ai/keys)"

import typesafe_sdk

print("typesafe-sdk", typesafe_sdk.__version__, "ready")

typesafe-sdk 0.7.5 ready


In [2]:
PRICE_PER_MTOK = 0.042  # dollars per million input tokens; output tokens are free


def cost_usd(response):
    """What this call cost, in dollars."""
    return response.usage.input_tokens / 1_000_000 * PRICE_PER_MTOK


def show(response):
    """Pretty-print every answer in a response, plus tokens and cost."""
    print(f"model={response.model}  input_tokens={response.usage.input_tokens}  cost=${cost_usd(response):.6f}")
    for qid, a in response.answers.items():
        if a.type == "noul":
            print(f"  {qid:<28} noul={a.noul:.2f}")
        elif a.type == "choice":
            print(f"  {qid:<28} choice={a.choice!r}  confidence={a.confidence:.2f}")
            print(f"  {'':<28} probabilities={a.probabilities}")
        elif a.type == "score":
            print(f"  {qid:<28} score={a.score:.2f}  confidence={a.confidence:.2f}")
            print(f"  {'':<28} probabilities={a.probabilities}")
        else:
            print(f"  {qid:<28} {a}")

## First, untangle the three numbers

A Score answer carries `probabilities`, `score`, AND `confidence`, and the most common
confusion in the whole API is what each one means. The key sentence:

> **`probabilities` IS the answer. `score` and `confidence` are two different one-number
> summaries of it, built to answer two different questions in your code.**

The analogy that makes it stick: imagine asking **100 experts** to place a bug report on
your three severity levels.

- `probabilities` = **the vote split.** 0 experts said "cosmetic", 57 said "workaround
  exists", 43 said "blocking". This is the raw belief; everything else is computed from it.
  (Those are the docs' recorded numbers for the ticket below; your live run may wiggle by a
  point, which is itself a good thing to show the class.)
- `score` = **the average position on the ruler.** Each level number weighted by its votes:
  `0*0.00 + 1*0.57 + 2*0.43 = 1.43`. One number to rank by or threshold.
- `confidence` = **how much the experts agreed.** Votes piled on one level means near 1;
  votes spread out means near 0. Here it is 0.35: a genuinely torn panel.

The division of labor in code:

```python
severity = response.scores["severity"]

if severity.confidence < 0.5:        # confidence: whether to trust the read at all
    send_to_review(ticket)
elif severity.score > 1.5:           # score: WHERE on the ruler (rank, threshold)
    escalate(ticket)

runner_up = sorted(severity.probabilities.values())[-2]   # probabilities: full detail
```

For the other primitives this collapses: a **Choice** has no score; the winning option
(`choice`) plays that role next to the same `probabilities` and `confidence`. A **Noul**
is a single probability, so there is nothing to summarize: a value near 0.5 IS low
confidence.

Let's see all three numbers come out of one live call, and verify that `score` really is
just the probability-weighted mean.

In [3]:
from typesafe_sdk import Score, TypeSafeClient

safari_bug = (
    "The export button crashes the settings page in Safari. It works in Chrome, "
    "but a few of our customers only use Safari."
)

with TypeSafeClient() as client:
    r = client.system_one(
        state=safari_bug,
        questions={
            "severity": Score(
                instructions="How severe is the reported issue?",
                criteria=[
                    "Cosmetic; no impact to functionality",
                    "Broken or degraded feature, but workaround exists",
                    "Blocking issue; no workaround exists",
                ],
            ),
        },
    )

a = r.scores["severity"]
print("probabilities (the vote split):", a.probabilities)
print(f"score         (avg position):  {a.score}")
print(f"confidence    (agreement):     {a.confidence}")

# score is nothing mysterious: recompute it from the probabilities ourselves
hand_computed = sum(level * p for level, p in a.probabilities.items())
print(f"\nweighted mean by hand:         {hand_computed:.2f}  ->  matches `score`")
assert abs(hand_computed - a.score) < 0.02

probabilities (the vote split): {0: 0.0, 1: 0.58, 2: 0.42}
score         (avg position):  1.42
confidence    (agreement):     0.37

weighted mean by hand:         1.42  ->  matches `score`


Why the model is split here, by the way: both readings of this ticket are TRUE. Chrome
users have a workaround (level 1), the Safari-only customers do not (level 2). The split
is honesty, not failure, and the fractional score ranks the bug between the two levels.

### Why score alone is not enough

Here is the trap, and the entire reason `confidence` exists: **different vote splits can
produce the same score.**

| probabilities | score | confidence | what is really going on |
| --- | --- | --- | --- |
| `{0: 0.0, 1: 1.0, 2: 0.0}` | 1.0 | 1.0 | everyone agrees: solidly level 1 |
| `{0: 0.5, 1: 0.0, 2: 0.5}` | 1.0 | 0.0 | total disagreement between the two extremes |

Same score, opposite situations. The first you can act on; the second should go to a
human. A threshold on `score` alone cannot tell them apart. We verify this table with the
exact API formula later in this notebook (see `score_confidence`), including the in-between
case `{0: 0.0, 1: 0.5, 2: 0.5}`, which keeps confidence 0.25 because the disagreement is
between NEIGHBORING levels rather than opposite ends.

## Watch confidence track ambiguity

Same question, three states: one clear, one mixed, one that the options do not cover at all.

In [4]:
from typesafe_sdk import Choice, TypeSafeClient

DEPARTMENT = Choice(
    instructions="Which team should handle this?",
    criteria={
        "returns": "Exchanges, wrong or damaged items",
        "shipping": "Delivery status, delays, lost packages",
        "billing": "Charges, invoices, payment problems",
    },
)

states = {
    "clear": "My invoice shows a charge I never made.",
    "mixed": "The shoes came in the wrong size and I was also charged twice.",
    "uncovered": "Do you have this jacket in green? Asking before I order.",
}

with TypeSafeClient() as client:
    for label, s in states.items():
        r = client.system_one(state=s, questions={"department": DEPARTMENT})
        a = r.choices["department"]
        print(f"{label:<10} choice={a.choice:<9} confidence={a.confidence:.2f}  probabilities={a.probabilities}")

clear      choice=billing   confidence=1.00  probabilities={'returns': 0.0, 'shipping': 0.0, 'billing': 1.0}


mixed      choice=returns   confidence=0.55  probabilities={'returns': 0.7, 'shipping': 0.0, 'billing': 0.3}


uncovered  choice=returns   confidence=0.83  probabilities={'returns': 0.89, 'shipping': 0.08, 'billing': 0.03}


The `uncovered` row is the one to study: the question has no good option, so the model
spreads probability and confidence collapses. Low confidence on a Choice usually means one
of three things: none of the options fit, two options genuinely overlap for this state, or
the state does not say enough. (This is also why adding an `other` option is kind to both
the model and your metrics.)

## The formulas, verified live

Confidence is not magic: it is a published formula over the `probabilities` you already
have. Let's implement all three and check them against the API's own numbers.

**Choice** with $n$ options, where $p_{max}$ is the top probability. Measures how far the
winner sits above an even split:

$$\text{confidence} = \frac{p_{max} - 1/n}{1 - 1/n}$$

**Score** with $n$ levels, where $m$ is the most likely level. Probability on a *nearby*
level hurts confidence less than probability far away:

$$\text{confidence} = \max\left(0,\ 1 - \frac{\sum_i p_i\,|i - m|}{\text{MAD}_{unif}}\right),
\qquad \text{MAD}_{unif} = \frac{1}{n}\sum_i \left|i - \frac{n-1}{2}\right|$$

**Noul** has no confidence field (two outcomes, the value says it all), but if you want to
gate Nouls and Choices with the same code, use the distance from a coin flip:

$$\text{confidence} = |2p - 1|$$

In [5]:
def choice_confidence(probabilities: dict) -> float:
    ps = list(probabilities.values())
    n = len(ps)
    return (max(ps) - 1 / n) / (1 - 1 / n)


def score_confidence(probabilities: dict) -> float:
    levels = sorted(probabilities)
    ps = [probabilities[level] for level in levels]
    n = len(ps)
    m = ps.index(max(ps))
    spread = sum(p * abs(i - m) for i, p in enumerate(ps))
    even_spread = sum(abs(i - (n - 1) / 2) for i in range(n)) / n
    return max(0.0, 1 - spread / even_spread)


def noul_confidence(p: float) -> float:
    return abs(2 * p - 1)

In [6]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

state = "The export crashes in Safari but works in Chrome. A few customers only use Safari."

with TypeSafeClient() as client:
    r = client.system_one(
        state=state,
        questions={
            "severity": Score(
                instructions="How severe is the reported issue?",
                criteria=[
                    "Cosmetic; no impact to functionality",
                    "Broken or degraded feature, but workaround exists",
                    "Blocking issue; no workaround exists",
                ],
            ),
            "area": Choice(
                instructions="Which product area does this concern?",
                criteria={"export": None, "auth": None, "settings_ui": None, "performance": None},
            ),
            "browser_specific": Noul(instructions="Is the issue specific to one browser?"),
        },
    )

sev, area, noul = r.scores["severity"], r.choices["area"], r.nouls["browser_specific"]

print(f"Score  API confidence={sev.confidence:.4f}   ours={score_confidence(sev.probabilities):.4f}")
print(f"Choice API confidence={area.confidence:.4f}   ours={choice_confidence(area.probabilities):.4f}")
print(f"Noul   value={noul.noul:.2f}                DIY confidence={noul_confidence(noul.noul):.4f}")

assert abs(sev.confidence - score_confidence(sev.probabilities)) < 0.02
assert abs(area.confidence - choice_confidence(area.probabilities)) < 0.02
print("\nformulas verified against the API")

Score  API confidence=0.4900   ours=0.4900
Choice API confidence=1.0000   ours=1.0000
Noul   value=0.93                DIY confidence=0.8600

formulas verified against the API


### The same-score table, verified with the real formula

Earlier we claimed that score 1.0 can hide opposite situations. Now that we have the exact
API formula implemented, prove it:

In [7]:
cases = {
    "everyone agrees on level 1":      {0: 0.0, 1: 1.0, 2: 0.0},
    "torn between opposite extremes":  {0: 0.5, 1: 0.0, 2: 0.5},
    "torn between NEIGHBORING levels": {0: 0.0, 1: 0.5, 2: 0.5},
}

print(f"{'case':<36}{'score':>7}{'confidence':>12}")
for label, probs in cases.items():
    score = sum(level * p for level, p in probs.items())
    conf = score_confidence(probs)
    print(f"{label:<36}{score:>7.2f}{conf:>12.2f}")

case                                  score  confidence
everyone agrees on level 1             1.00        1.00
torn between opposite extremes         1.00        0.00
torn between NEIGHBORING levels        1.50        0.25


Three lessons in one table:

1. Rows 1 and 2: **same score, confidence 1.0 vs 0.0.** Act on the first, escalate the second.
2. Rows 2 and 3: **same amount of disagreement, different confidence.** The Score formula
   is distance-aware: probability on a neighboring level (0.25) hurts far less than
   probability on the opposite end (0.0), because being torn between "workaround exists"
   and "blocking" is a much smaller disagreement than between "cosmetic" and "blocking".
   (A Choice, whose options have no order, would give both rows the same confidence.)
3. The practical rule that falls out: **threshold on `score` for position, gate on
   `confidence` for trust, and read `probabilities` when the two disagree.**

Because the full `probabilities` always comes back, `confidence` is just a sensible default.
Two alternatives the docs recommend trying for Choices:

- **Top probability** ($p_{max}$): reads directly as "how likely is the selected option",
  but its meaning depends on the option count, so set thresholds per question.
- **Top-to-second ratio** ($p_{max}/p_{second}$): how clearly the winner beats the runner-up.
  Many real decisions come down to the top two candidates.

And one anti-pattern from TypeSafe's own troubleshooting guide: if all you ever do is pick
the best option, you do not need a confidence threshold at all; just take `choice`.
Thresholds earn their keep when some answers should NOT be acted on.

## The three-path pattern

The architectural payoff. Divide confidence into three ranges, each with its own behavior:

| Range | Behavior |
| --- | --- |
| High | Act automatically |
| Medium | Proceed with caution: confirm with the user, flag for review |
| Low | Do not act: route to a human, ask for clarification, fall back |

Crucially, **thresholds scale with risk**. One system, different gates per action: showing a
balance at 0.6 confidence is fine (worst case, a wrong screen); approving a transfer needs
0.85+ and still confirms. Your code encodes the risk tolerance, visibly, in review-able
constants.

In [8]:
from typesafe_sdk import Choice, TypeSafeClient

INTENT = Choice(
    instructions="What is the user trying to do?",
    criteria={
        "check_balance": "View account balance",
        "approve_transfer": "Approve the pending withdrawal request",
        "support": "Get help with an issue",
    },
)

commands = [
    "what's my balance",
    "yes go ahead and approve the transfer",
    "approve it I guess? actually wait",
    "the app charged me twice",
    "do the thing we discussed",
]


def route(command: str, answer) -> str:
    if answer.confidence < 0.5:
        return "route_to_human"
    if answer.choice == "check_balance":
        return "show_balance"                      # low stakes, 0.5 is enough
    if answer.choice == "approve_transfer":
        if answer.confidence > 0.85:
            return "approve_transfer"              # high stakes, high confidence
        return "ask_user_to_confirm"               # high stakes, medium confidence
    return "open_support_ticket"


with TypeSafeClient() as client:
    for cmd in commands:
        a = client.system_one(state=cmd, questions={"intent": INTENT}).choices["intent"]
        print(f"{cmd!r:<42} -> {route(cmd, a):<22} (choice={a.choice}, confidence={a.confidence:.2f})")

"what's my balance"                        -> show_balance           (choice=check_balance, confidence=1.00)


'yes go ahead and approve the transfer'    -> approve_transfer       (choice=approve_transfer, confidence=1.00)
'approve it I guess? actually wait'        -> approve_transfer       (choice=approve_transfer, confidence=0.94)


'the app charged me twice'                 -> open_support_ticket    (choice=support, confidence=1.00)
'do the thing we discussed'                -> route_to_human         (choice=support, confidence=0.49)


## How stable are the numbers?

A classifier you put in production should give the same answer tomorrow. System One models
are designed to be self-consistent across repeated evaluations. Test it on an ambiguous
input, where drift would show first:

In [9]:
from typesafe_sdk import Noul, TypeSafeClient

ambiguous = "Are you a bot?"  # hinted, never asked: a genuine mid-probability case

values = []
with TypeSafeClient() as client:
    for _ in range(5):
        r = client.system_one(
            state=ambiguous,
            questions={"wants_human": Noul(instructions="Is the customer asking for a human agent?")},
        )
        values.append(r.nouls["wants_human"].noul)

print("five runs:", values)
print("spread:   ", round(max(values) - min(values), 3))

five runs: [0.37, 0.35, 0.4, 0.37, 0.36]
spread:    0.05


A small spread on a genuinely ambiguous input is what you want to see: the uncertainty lives
in the *value* (near 0.5), not in run-to-run randomness.

## Exercises

1. Write `top_to_second(probabilities)` and re-run the banking router gating on ratio > 3
   instead of confidence. Which commands change their route?
2. For the Score formula, compute confidence of `{0: 0.5, 1: 0.0, 2: 0.5}` and
   `{0: 0.0, 1: 0.5, 2: 0.5}` by hand. Why does the second one score higher? Verify with
   `score_confidence`.
3. Calibration is testable: collect 20 Nouls with known answers, bucket the values, and
   compare bucket frequency with bucket midpoint.